# Notebook 02: Limpeza e Engenharia de Atributos
Este notebook é focado exclusivamente em preparar a base para modelagem. Iremos tratar valores nulos com base na regra de negócio e realizar as transformações categóricas (Enconding).

## 1. Carregamento e Remoção de Ruídos
Começamos da base crua novamente e descartamos colunas desnecessárias.

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('../data/processed', exist_ok=True)

df = pd.read_csv('../data/raw/passenger_survey_balanced.csv')

# Remover colunas de texto livre ou IDs que não serão processadas (se houverem)
# df.drop(columns=['passenger_id', 'comments'], inplace=True, errors='ignore')

print(f"Shape inicial: {df.shape}")


Shape inicial: (57514, 145)


## 2. Tratamento de Nulos Estruturais (O Pulo do Gato)
Tratamento condicional baseado nas colunas `_is_applicable`. Serviços não utilizados recebem `-1`.

In [2]:
def treat_structural_nulls(data):
    df_clean = data.copy()
    
    # Colunas de aplicabilidade
    applicability_cols = [col for col in df_clean.columns if str(col).endswith('_is_applicable')]
    
    for app_col in applicability_cols:
        eval_col = app_col.replace('_is_applicable', '')
        if eval_col in df_clean.columns:
            # Se is_applicable == 0, preenchemos com -1
            mask_not_applicable = (df_clean[app_col] == 0) & (df_clean[eval_col].isnull())
            df_clean.loc[mask_not_applicable, eval_col] = -1
            
    # Tratamento dos NaNs remanescentes (aplicabilidade == 1 ou sem flag)
    for col in df_clean.columns:
        if df_clean[col].isnull().any():
            if pd.api.types.is_numeric_dtype(df_clean[col]):
                df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            else:
                df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
                
    return df_clean

nulos_iniciais = df.isnull().sum().sum()
print(f"Nulos na entrada: {nulos_iniciais}")

df_cleaned = treat_structural_nulls(df)

nulos_tratados = df_cleaned.isnull().sum().sum()
print(f"Nulos após tratamento: {nulos_tratados}")


Nulos na entrada: 3323783
Nulos após tratamento: 0


## 3. Encoding (Transformação Categórica)
Convertendo variáveis textuais para formatos numéricos através de One-Hot Encoding.

In [3]:
# Identificar colunas categóricas (texto/object)
cat_cols = df_cleaned.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Colunas para aplicar One-Hot Encoding ({len(cat_cols)}): {cat_cols}")

# Aplicar get_dummies
df_encoded = pd.get_dummies(df_cleaned, columns=cat_cols, drop_first=True)

# Garantir que todos os booleanos gerados sejam inteiros (0 e 1)
for c in df_encoded.columns:
    if df_encoded[c].dtype == 'bool':
        df_encoded[c] = df_encoded[c].astype(int)

print(f"Shape após o Encoding: {df_encoded.shape}")


Colunas para aplicar One-Hot Encoding (23): ['process', 'month', 'flight_type', 'connection', 'ticket_purchased_by', 'ticket_purchase_channel', 'transport_to_airport', 'has_disability', 'uses_assistive_device', 'requested_special_assistance', 'disembarkation_method_used', 'used_parking', 'checkin_method', 'nationality', 'gender', 'age_group', 'education', 'household_income', 'traveling_alone', 'number_of_companions', 'trip_purpose', 'trips_last_12_months', 'used_airport_before_last_12_months']


C:\Users\helio\AppData\Local\Temp\ipykernel_7912\4218695051.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_cleaned.select_dtypes(include=['object', 'category']).columns.tolist()


Shape após o Encoding: (57514, 252)


## 4. Verificação Final e Exportação
Garantindo ausência de NaNs e exportando para `data/processed/`.

In [4]:
# Verificação Final
assert df_encoded.isnull().sum().sum() == 0, "Ainda existem valores nulos na base!"
assert df_encoded.select_dtypes(exclude=[np.number]).shape[1] == 0, "Ainda existem colunas de texto/não-numéricas!"

print("Base aprovada em todas as checagens metodológicas.")

# Exportar
output_path = '../data/processed/df_model.csv'
df_encoded.to_csv(output_path, index=False)
print(f"Arquivo CSV finalizado e exportado com sucesso para: {output_path}")


Base aprovada em todas as checagens metodológicas.
Arquivo CSV finalizado e exportado com sucesso para: ../data/processed/df_model.csv
